# SessionRerank+ — Kaggle T4 Execution Notebook
> **v19 — fixed**: torchvision uninstall in install cell, Kaggle-local Gemma path auto-detect, correct cell order. Run all cells top-to-bottom. GPU T4 required.

In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NOT FOUND')
print('Python:', sys.version)

In [ ]:
import subprocess, sys
def run(cmd, **kw):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)

# 1. pin torch 2.4.1+cu121 (avoids cu128 ABI mismatch)
print('[install] pinning torch 2.4.1+cu121...')
r = run('pip install -q torch==2.4.1+cu121 torchaudio==2.4.1+cu121 --index-url https://download.pytorch.org/whl/cu121')

# 2. uninstall incompatible torchvision BEFORE any torch import
print('[install] removing incompatible torchvision...')
run('pip uninstall -y torchvision')

# 3. ensure setuptools
run('pip install -q setuptools')

# 4. project dependencies (no torchvision)
print('[install] installing project deps...')
r = run('pip install -q sentence-transformers==2.7.0 transformers==4.44.2 accelerate bitsandbytes rank-bm25 datasets huggingface_hub')
if r.returncode != 0: print('WARN dep install:', r.stderr[-300:])
print('[install] done')

In [ ]:
import torch
print(f'torch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
import subprocess, os
repo = '/kaggle/working/sr'
if not os.path.exists(repo):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/joyjeni/session-aware-toolbench-rerank', repo],
                   check=True)
    print('cloned to', repo)
else:
    subprocess.run(['git','-C',repo,'pull'], check=True)
    print('pulled at', repo)
os.chdir(repo)
print('cwd:', os.getcwd())

In [ ]:
import subprocess, os
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
                    capture_output=True, text=True, cwd='/kaggle/working/sr')
if r.returncode != 0: print('pkg install stderr:', r.stderr[-500:])
print('[install pkg] done')

In [ ]:
import os, glob

# Auto-detect Kaggle-local Gemma model path (no HF token needed)
GEMMA_PATH = None

# Search for Gemma in Kaggle input
for pattern in [
    '/kaggle/input/gemma/transformers/2b-it/*',
    '/kaggle/input/gemma-2/transformers/gemma-2-2b-it/*',
    '/kaggle/input/gemma/*/2b-it/*',
]:
    matches = sorted(glob.glob(pattern))
    if matches:
        GEMMA_PATH = matches[-1]  # latest version
        print(f'[gemma] found local model at: {GEMMA_PATH}')
        break

if GEMMA_PATH is None:
    # Fall back to HF (requires HF_TOKEN if gated)
    GEMMA_PATH = 'google/gemma-2-2b-it'
    print(f'[gemma] WARNING: no local model found, using HF: {GEMMA_PATH}')
    try:
        from kaggle_secrets import UserSecretsClient
        import os
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('[gemma] HF_TOKEN set from Kaggle Secrets')
    except BaseException:
        print('[gemma] WARNING: HF_TOKEN not set, gated model may fail')

print(f'[gemma] using: {GEMMA_PATH}')

In [ ]:
import os, subprocess, sys
data_dir = '/kaggle/working/sr/data/toolbench_hf'
if not os.path.exists(data_dir):
    print('[download] fetching ToolBench from HuggingFace...')
    r = subprocess.run(
        [sys.executable, '-m', 'sessionrerank.data.download',
         '--out', '/kaggle/working/sr/data/toolbench_hf'],
        cwd='/kaggle/working/sr', capture_output=True, text=True
    )
    print(r.stdout[-1000:] if r.stdout else '')
    if r.returncode != 0:
        print('stderr:', r.stderr[-500:])
else:
    print('[download] data already present at', data_dir)

In [ ]:
import pathlib
parquets = list(pathlib.Path('/kaggle/working/sr/data/toolbench_hf').rglob('*.parquet'))
print(f'found {len(parquets)} parquet files')
for p in sorted(parquets)[:5]: print(' ', p)

In [ ]:
import subprocess, sys, os, time

# Use auto-detected Kaggle local Gemma path (set in cell-gemma-path above)
gemma_model = GEMMA_PATH

cmd = [
    sys.executable, '-m', 'sessionrerank.eval.run_all',
    '--device', 'cuda',
    '--encoder', 'sentence-transformers/all-MiniLM-L6-v2',
    '--ce', 'cross-encoder/ms-marco-MiniLM-L-6-v2',
    '--ce-topk', '100',
    '--gemma', gemma_model,
    '--gemma-rerank-topk', '50', '--gemma-batch-size', '4',
    '--gemma-4bit',
    '--free-embed-after-encode',
    '--resume',
    '--out', 'results/metrics.json',
]
print('[run_all] starting... (~30-50 min on T4)')
print('[run_all] gemma model:', gemma_model)
t0 = time.time()
r = subprocess.run(cmd, cwd='/kaggle/working/sr')
print(f'[run_all] exit={r.returncode} elapsed={time.time()-t0:.0f}s')

In [ ]:
import json, pathlib
p = pathlib.Path('/kaggle/working/sr/results/metrics.json')
if p.exists():
    data = json.loads(p.read_text())
    print(json.dumps(data, indent=2))
else:
    print('[results] no results file found')
    # show partial checkpoint if any
    ckpts = list(pathlib.Path('/kaggle/working/sr/results').glob('*.json')) if pathlib.Path('/kaggle/working/sr/results').exists() else []
    print(f'[results] checkpoint files: {ckpts}')